# 📊 Clase 3 — Detección y Tratamiento de Outliers (R)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 23 de abril de 2026

---

### Objetivos

1. **Detectar** outliers con IQR, z-score modificado (MAD) y métodos multivariados.
2. **Clasificar** y **contextualizar** la detección por segmento.
3. **Tratar** con winsorización, transformación log, capping y variable indicadora.
4. **Evaluar** el impacto sobre distribución y scoring.

> **Paso previo:** Montar Drive en Python, cambiar runtime a R, ejecutar `source()` del setup.


---\n## 0. Montar Google Drive (runtime Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado. Cambiar runtime a R y continuar.")


---\n## 1. Configuración del entorno R

In [ ]:
source("/content/drive/MyDrive/R_Colab/setup_R_colab.R")

paq <- c("tidyverse", "gridExtra", "DescTools", "isotree", "pROC")
faltantes <- paq[!(paq %in% installed.packages()[, "Package"])]
if (length(faltantes) > 0) {
  cat(sprintf("📦 Instalando: %s\n", paste(faltantes, collapse = ", ")))
  install.packages(faltantes, quiet = TRUE)
}

suppressPackageStartupMessages({
  library(tidyverse); library(gridExtra); library(pROC)
})

# DescTools para Winsorize
desc_ok <- tryCatch({ library(DescTools); TRUE }, error = function(e) FALSE)

theme_set(theme_minimal(base_size = 12) +
          theme(plot.title = element_text(face = "bold", size = 14),
                legend.position = "bottom"))

col_good <- "#2a9d8f"; col_bad <- "#e76f51"; col_acc <- "#f4a261"

cat("\n✅ Entorno R configurado\n")


---\n## 2. Carga del dataset

In [ ]:
url_data <- "https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff"
df <- tryCatch(read_csv(url_data, show_col_types = FALSE),
               error = function(e) read_csv("/content/drive/MyDrive/datasets/german_credit.csv", show_col_types = FALSE))

num_vars <- c("duration", "credit_amount", "installment_commitment",
              "residence_since", "age", "existing_credits", "num_dependents")

cat(sprintf("✅ Dataset: %d filas × %d columnas\n", nrow(df), ncol(df)))


---\n## 3. Método IQR (Tukey Fences)

In [ ]:
detect_iqr <- function(x, k = 1.5) {
  Q1 <- quantile(x, 0.25, na.rm = TRUE)
  Q3 <- quantile(x, 0.75, na.rm = TRUE)
  iqr <- Q3 - Q1
  lower <- Q1 - k * iqr; upper <- Q3 + k * iqr
  mask <- x < lower | x > upper
  list(mask = mask, lower = lower, upper = upper, n = sum(mask))
}

cat("═══════════════════════════════════════════════════\n")
cat("DETECCIÓN DE OUTLIERS POR IQR (k=1.5)\n")
cat("═══════════════════════════════════════════════════\n")
cat(sprintf("  %-28s %8s %7s %10s %10s\n", "Variable", "Outliers", "%", "Lím.inf.", "Lím.sup."))
cat(paste0("  ", strrep("-", 65), "\n"))

for (v in num_vars) {
  res <- detect_iqr(df[[v]])
  pct <- res$n / nrow(df) * 100
  flag <- ifelse(pct > 5, "  ⚠", "")
  cat(sprintf("  %-28s %8d %6.1f%% %10.1f %10.1f%s\n",
              v, res$n, pct, res$lower, res$upper, flag))
}


In [ ]:
options(repr.plot.width = 16, repr.plot.height = 5)

key_vars <- c("credit_amount", "age", "duration")
plots_box <- map(key_vars, function(v) {
  data <- df[[v]]
  res <- detect_iqr(data)
  tibble(value = data, outlier = res$mask) %>%
    ggplot(aes(x = v, y = value)) +
    geom_boxplot(fill = col_good, alpha = 0.4, width = 0.4, outlier.shape = NA) +
    geom_jitter(data = . %>% filter(outlier), aes(color = "Outlier"),
                width = 0.08, size = 1.5, alpha = 0.6) +
    geom_hline(yintercept = c(res$lower, res$upper), linetype = "dashed",
               color = col_acc, linewidth = 0.5) +
    scale_color_manual(values = c("Outlier" = col_bad)) +
    labs(title = sprintf("%s (%d outliers)", v, res$n), x = NULL, y = v) +
    theme(legend.title = element_blank())
})

grid.arrange(grobs = plots_box, ncol = 3,
             top = grid::textGrob("Detección IQR (k=1.5)", gp = grid::gpar(fontface = "bold", fontsize = 13)))


### 📝 Interpretación

`credit_amount` presenta ~53 outliers (5.3%) — clientes de alto monto, no errores. `num_dependents` muestra falsos positivos por su escala discreta. El IQR resulta inadecuado para variables de baja cardinalidad.

---
## 4. Z-score modificado (MAD)


In [ ]:
detect_zscore_mad <- function(x, threshold = 3.5) {
  med <- median(x, na.rm = TRUE)
  mad_val <- mad(x, constant = 1, na.rm = TRUE)
  if (mad_val == 0) return(list(mask = rep(FALSE, length(x)), scores = rep(0, length(x))))
  m_scores <- 0.6745 * (x - med) / mad_val
  mask <- abs(m_scores) > threshold
  list(mask = mask, scores = m_scores, n = sum(mask))
}

cat("═══════════════════════════════════════════════════\n")
cat("Z-SCORE MODIFICADO (MAD) — |M| > 3.5\n")
cat("═══════════════════════════════════════════════════\n")
cat(sprintf("  %-28s %8s %7s %8s\n", "Variable", "Outliers", "%", "M_max"))

for (v in num_vars) {
  res <- detect_zscore_mad(df[[v]])
  pct <- res$n / nrow(df) * 100
  m_max <- max(abs(res$scores))
  cat(sprintf("  %-28s %8d %6.1f%% %8.2f\n", v, res$n, pct, m_max))
}

# Comparación IQR vs MAD en credit_amount
ca <- df$credit_amount
iqr_res <- detect_iqr(ca)
mad_res <- detect_zscore_mad(ca)
cat(sprintf("\n  IQR: %d outliers | MAD: %d outliers | Coinciden: %d\n",
            iqr_res$n, mad_res$n, sum(iqr_res$mask & mad_res$mask)))


### 📝 Interpretación

El z-score MAD detecta menos outliers que el IQR en `credit_amount` gracias a su mayor robustez (breakdown 50%). En variables asimétricas como montos monetarios, la MAD evita los falsos positivos que el IQR genera al inflarse con la asimetría.

---
## 5. IQR contextualizado por segmento


In [ ]:
df <- df %>%
  mutate(tramo_edad = cut(age, breaks = c(18, 25, 35, 45, 55, Inf),
                           labels = c("18-25", "26-35", "36-45", "46-55", "56+")))

cat("═══════════════════════════════════════════════════\n")
cat("IQR CONTEXTUALIZADO: credit_amount por tramo\n")
cat("═══════════════════════════════════════════════════\n")

df %>%
  group_by(tramo_edad) %>%
  summarise(
    n = n(),
    Q1 = quantile(credit_amount, 0.25),
    Q3 = quantile(credit_amount, 0.75),
    IQR = IQR(credit_amount),
    lim_sup = Q3 + 1.5 * IQR,
    outliers = sum(credit_amount > lim_sup | credit_amount < Q1 - 1.5 * IQR),
    pct = round(outliers / n * 100, 1),
    .groups = "drop"
  ) %>%
  print()


---\n## 6. Estrategias de tratamiento

In [ ]:
ca <- df$credit_amount

# T1: Winsorización P1/P99 — compatible con DescTools antigua y nueva.
# DescTools refactorizó Winsorize() en la versión 0.99.60: el argumento
# "probs" fue reemplazado por "val" que recibe el vector de umbrales ya
# calculados. El patrón tryCatch intenta primero la API nueva y, si falla
# (API vieja o paquete no disponible), cae al cálculo manual equivalente.
ca_wins <- tryCatch(
  Winsorize(ca, val = quantile(ca, probs = c(0.01, 0.99), na.rm = TRUE)),
  error = function(e) {
    p01 <- quantile(ca, 0.01, na.rm = TRUE)
    p99 <- quantile(ca, 0.99, na.rm = TRUE)
    pmin(pmax(ca, p01), p99)
  }
)

# T2: Log
ca_log <- log1p(ca)

# T3: Capping P5/P95
p5  <- quantile(ca, 0.05, na.rm = TRUE)
p95 <- quantile(ca, 0.95, na.rm = TRUE)
ca_cap <- pmin(pmax(ca, p5), p95)

# Comparación
options(repr.plot.width = 16, repr.plot.height = 8)
df_treat <- bind_rows(
  tibble(value = ca,       metodo = "Original"),
  tibble(value = ca_wins,  metodo = "Winsorizado P1/P99"),
  tibble(value = ca_log,   metodo = "log(1+x)"),
  tibble(value = ca_cap,   metodo = "Capping P5/P95")
) %>%
  mutate(metodo = factor(metodo, levels = c("Original", "Winsorizado P1/P99", "log(1+x)", "Capping P5/P95")))

ggplot(df_treat, aes(x = value, fill = metodo)) +
  geom_histogram(bins = 40, color = "white", alpha = 0.8) +
  facet_wrap(~ metodo, scales = "free", ncol = 2) +
  scale_fill_manual(values = c(col_good, col_acc, "#8B5CF6", "#3B82F6")) +
  labs(title = "Estrategias de tratamiento de outliers — credit_amount",
       x = "Valor", y = "Frecuencia") +
  theme(legend.position = "none", strip.text = element_text(face = "bold"))


In [ ]:
# Estadísticos comparativos
cat("\n  Estadísticos comparativos:\n")
cat(sprintf("  %-25s %10s %10s %10s %10s\n", "Método", "Media", "Mediana", "Desvío", "Asimetría"))
cat(paste0("  ", strrep("-", 68), "\n"))

skew <- function(x) { n <- length(x); m <- mean(x); s <- sd(x); n/((n-1)*(n-2)) * sum(((x-m)/s)^3) }

for (info in list(
  list("Original", ca), list("Winsorizado P1/P99", ca_wins),
  list("log(1+x)", ca_log), list("Capping P5/P95", ca_cap))) {
  nm <- info[[1]]; v <- info[[2]]
  cat(sprintf("  %-25s %10.1f %10.1f %10.1f %10.3f\n",
              nm, mean(v), median(v), sd(v), skew(v)))
}


### 📝 Interpretación

- **Winsorización:** Cambio mínimo en media/mediana, reduce desvío y asimetría de forma conservadora.
- **Log(1+x):** Reduce la asimetría drásticamente (de ~2.0 a ~0.01). Ideal para variables monetarias.
- **Capping P5/P95:** Más agresivo; útil cuando se requieren rangos predefinidos (scoring regulatorio).

---
## 7. Impacto en scoring


In [ ]:
cat("═══════════════════════════════════════════════════\n")
cat("IMPACTO EN SCORING: AUC-ROC\n")
cat("═══════════════════════════════════════════════════\n")

target <- as.integer(df$class == "bad")

get_auc <- function(X, y) {
  fit <- glm(y ~ ., data = data.frame(X, y = y), family = binomial)
  probs <- predict(fit, type = "response")
  as.numeric(auc(roc(y, probs, quiet = TRUE)))
}

X_base <- df[num_vars]
auc_base <- get_auc(X_base, target)

mask_out <- detect_iqr(ca)$mask

strategies <- list(
  list("Sin tratamiento", X_base, target),
  list("Eliminar outliers", X_base[!mask_out, ], target[!mask_out]),
  list("Winsorizar P1/P99", X_base %>% mutate(credit_amount = ca_wins), target),
  list("Log(credit_amount)", X_base %>% mutate(credit_amount = ca_log), target),
  list("Capping P5/P95", X_base %>% mutate(credit_amount = ca_cap), target),
  list("Agregar flag outlier", X_base %>% mutate(is_outlier = as.integer(mask_out)), target)
)

for (s in strategies) {
  a <- get_auc(s[[2]], s[[3]])
  delta <- a - auc_base
  cat(sprintf("  %-25s  AUC = %.4f  (Δ = %+.4f)\n", s[[1]], a, delta))
}


### 📝 Interpretación

Los resultados confirman que eliminar outliers genuinos degrada la performance. Las transformaciones (log, winsorización) preservan o mejoran el AUC al reducir la influencia de valores extremos sin perder información.

---
## 8. Resumen


In [ ]:
cat("╔══════════════════════════════════════════════════════════════════╗\n")
cat("║          RESUMEN — OUTLIERS EN DATOS FINANCIEROS (R)          ║\n")
cat("╠══════════════════════════════════════════════════════════════════╣\n")
cat("║  Paquetes clave:                                                ║\n")
cat("║    · Base R: boxplot.stats(), IQR(), mad()                     ║\n")
cat("║    · DescTools: Winsorize()                                     ║\n")
cat("║    · isotree: isolation.forest() (alternativa multivariada)     ║\n")
cat("║    · tidyverse: group_by() para IQR contextualizado            ║\n")
cat("║  Regla de oro: NUNCA eliminar outliers genuinos en finanzas     ║\n")
cat("╚══════════════════════════════════════════════════════════════════╝\n")


---\n> **Dr. Darío Ezequiel Díaz** · MMD UTN FRRo · 2026